<p align="center">
  <a href="https://asterisk.coop" target="_blank"><img src="https://raw.githubusercontent.com/asterisk-labs/asterisk-labs.github.io/main/brand/assets/svg/logo-horizontal-color.svg" height="58" alt="Asterisk Labs"></a>
  &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;
  <a href="https://2026.cloudnativegeo.org" target="_blank"><img src="https://us-previewenginepublic.nimbuspop.com/image/BACKSTAGE/200860000000039005?cli-msg=eyJtb2R1bGUiOiJFdmVudEltYWdlUmVzb3VyY2UiLCJ0eXBlIjoxLCJwb3J0YWxJZCI6IjkwNTk5NDA2NiIsInN1YlJlc291cmNlSWQiOiIyMDA4NjAwMDAwMDAwMDIwNzIiLCJpZCI6IjIwMDg2MDAwMDAwMDAzOTAwNSJ9" height="66" alt="CNG Forum 2026"></a>
</p>

<p align="center">
  <a href="https://github.com/ESA-PhiLab/Major-TOM/blob/refactor/v1/docs/workshop/workshop.ipynb" target="_blank"><img src="https://img.shields.io/badge/View_on-GitHub-181717?logo=github" alt="View on GitHub"></a>
  &nbsp;
  <a href="https://colab.research.google.com/github/ESA-PhiLab/Major-TOM/blob/refactor/v1/docs/workshop/workshop.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"></a>
</p>

<p align="center"><font size="7"><b>BYOT (Build Your Own TOM)</b></font></p>
<p align="center"><font size="5">AI Dataset Building Crash Course</font></p>
<p align="center"><b>Workshop</b><br>Tue, Oct 6, 2026, 9:00 AM–12:30 PM (MDT)<br>Superior B</p>

### **Description**

In this hands-on workshop, you will turn public satellite data into an AI-ready dataset that others can discover, query, and use.

**PART I. INTRODUCTION (30 MIN)**  
We begin with a short tour of the tools and design choices behind the workflow. You will see how satellite imagery becomes a Major TOM dataset packaged with TACO for cloud access and AI training. We will then turn your interests into a dataset idea for Part II.

**PART II. HANDS-ON SESSION (2 HRS)**  
You will query public archives, select an area of interest, build a TACO dataset, and publish it to Source Cooperative, Hugging Face, or another platform. We will also discuss the strengths and limits of each data source. If you finish early, you can try an optional annotation track with IRIS.

**PART III. WRAP-UP (30 MIN)**  
We close with guidance on joining the Major TOM community and registering your dataset. You will leave with a published dataset or a clear plan to finish one. You can also give a one-minute lightning talk and invite others to collaborate.

<p align="center"><b>Speakers</b></p>
<center>
<table width="700" style="border:none; border-collapse:collapse; background:transparent; margin-left:auto; margin-right:auto;">
  <tr style="border:none; background:transparent;">
    <td width="350" align="center" valign="top" style="border:none; background:transparent; text-align:center;"><a href="https://github.com/csaybar" target="_blank"><img src="https://github.com/csaybar.png" width="144" height="144" alt="Cesar Aybar"></a><br><b>Cesar Aybar</b><br><small>Developer and Partner Scientist<br>Asterisk Labs</small></td>
    <td width="350" align="center" valign="top" style="border:none; background:transparent; text-align:center;"><a href="https://mikonvergence.github.io" target="_blank"><img src="https://raw.githubusercontent.com/mikonvergence/mikonvergence.github.io/master/imgs/profile-2.JPEG" width="144" height="144" alt="Mikolaj Czerkawski"></a><br><b>Mikolaj Czerkawski</b><br><small>Co-Founder &amp; Partner<br>Asterisk Labs</small></td>
  </tr>
</table>
</center>


# 1. Overview

Today, you will build and publish a small **AI-ready Earth observation dataset**.

You will:

1. choose data from a public satellite archive;
2. select an area and organize the samples on the Major TOM grid;
3. encode the raster tensors with Rumi;
4. package the samples and metadata as a TACO dataset;
5. load the result for AI training and compare read performance; and
6. publish the dataset to Source Cooperative or Hugging Face.

By the end, you will have a small Major TOM dataset—or a clear plan to finish one.


# 2. Setup

- **TACO** defines, writes, validates, and reads the dataset. Reading requires `taco-eo`; writing requires `taco-eo[writer]`.
- **Rumi** stores raster tensors compressed with GeoZL.
- **pystac-client**, **planetary-computer**, and **rasterio** find and download imagery.
- **huggingface_hub** publishes the finished dataset to Hugging Face.

Run the next cell once. Installation usually takes about one minute.


In [ ]:
%pip install -q "taco-eo[writer]" rumi-eo huggingface_hub \
    "majortom[build] @ git+https://github.com/ESA-PhiLab/Major-TOM@refactor/v1"

# 3. Choose your points

Bring your own points, or pick a class from our catalogue.

**Your own points.** Drag a CSV into the Files panel (folder icon on the left) and put its name in `MY_POINTS` in the cell below, or set `MY_POINTS = "upload"` to pick the file when the cell runs. Use a CSV with `lat` and `lon` columns (plus optional `name` or `title`, and `date`), or a GeoJSON of points. The dataset is named after the file. With a `date` column, each sample uses images from that date to `DAYS_AFTER` days later, so events at the same place on different dates become separate samples. Sentinel-2 starts in mid-2015; for older events choose `landsat-c2-l2` in section 4.

**Or a catalogue class.** Leave `MY_POINTS` empty and choose a class from our [Wikidata points catalog](https://source.coop/asterisk-labs/cng2026), which contains 370,042 locations across 41 classes. Use one of these exact values for `CLASS`.

| Class | Points | Class | Points |
|---|---:|---|---:|
| `airbase` | 3,086 | `mine` | 54,361 |
| `airport` | 24,889 | `nuclear_power_plant` | 381 |
| `atoll` | 114 | `oasis` | 191 |
| `caldera` | 106 | `oil_field` | 5,522 |
| `canal` | 50,622 | `oil_refinery` | 263 |
| `canyon` | 5,475 | `port` | 7,482 |
| `city` | 33,297 | `quarry` | 3,593 |
| `container_terminal` | 62 | `reservoir` | 27,904 |
| `crater` | 683 | `salt_evaporation_pond` | 229 |
| `dam` | 82,288 | `salt_pan` | 102 |
| `desert` | 2,662 | `ski_resort` | 1,769 |
| `dune` | 5,120 | `solar_power_plant` | 5,874 |
| `erg` | 196 | `stadium` | 13,358 |
| `fjord` | 2,171 | `stratovolcano` | 356 |
| `geyser` | 123 | `tepui` | 53 |
| `glacial_lake` | 1,552 | `theme_park` | 1,009 |
| `glacier` | 7,259 | `unesco_world_heritage` | 3,323 |
| `golf_course` | 1,936 | `volcano` | 2,852 |
| `horse_racing_venue` | 848 | `wind_farm` | 2,732 |
| `hydroelectric_power_station` | 9,536 |  |  |
| `lagoon` | 6,349 |  |  |
| `landfill` | 314 |  |  |

Later, set `N_SAMPLES` to the number of locations to include. Keep it at 50 or fewer during the workshop: each sample is a whole 10 km Major TOM cell, about 16 MB for Sentinel-2.


In [ ]:
import re
from pathlib import Path

import pandas as pd
import plotly.express as px

# Your own points: drag a CSV (lat, lon columns) or a GeoJSON of points into the Files panel
# (folder icon, left), then put its name here. "upload" opens a file picker instead; a URL also works.
# A "date" column (e.g. 2014-07-30) sets each point's time: its sample uses images from that day
# up to DAYS_AFTER days later. Points without a date use DATETIME from section 4.
MY_POINTS = ""          # e.g. "my_points.csv", "upload" or "https://.../points.geojson"
DAYS_AFTER = 60

# Or, with MY_POINTS empty: a class from the Wikidata catalogue (table above).
CLASS = "golf_course"
MAP_LIMIT = 2_000

if MY_POINTS:
    if MY_POINTS == "upload":                          # if no button appears, use the Files panel
        from google.colab import files
        MY_POINTS = next(iter(files.upload()))          # name of the uploaded file
    if not MY_POINTS.startswith("http") and not Path(MY_POINTS).exists():
        found = sorted(str(f) for f in Path().iterdir() if f.suffix.lower() in (".csv", ".geojson", ".json"))
        raise FileNotFoundError(f"{MY_POINTS!r} not found. Files here: {found}")
    if MY_POINTS.lower().endswith((".geojson", ".json")):
        import geopandas as gpd
        gdf = gpd.read_file(MY_POINTS).to_crs(4326)
        selected = pd.DataFrame(gdf.drop(columns="geometry")).assign(lon=gdf.geometry.x, lat=gdf.geometry.y)
    else:
        selected = pd.read_csv(MY_POINTS)
    if "name" not in selected:
        selected["name"] = selected["title"] if "title" in selected else selected.index.astype(str)
    if "date" in selected:                               # one time window per point
        start = pd.to_datetime(selected["date"], errors="coerce")
        end = start + pd.Timedelta(days=DAYS_AFTER)
        selected["datetime"] = start.dt.strftime("%Y-%m-%d") + "/" + end.dt.strftime("%Y-%m-%d")
    CLASS = re.sub(r"\s*\(\d+\)$", "", Path(MY_POINTS).stem)   # the dataset is named after the file
    if "class" not in selected:
        selected["class"] = CLASS
    print(f"{len(selected):,} points from {MY_POINTS}")
else:
    points = pd.read_parquet(
        "https://data.source.coop/asterisk-labs/cng2026/points.parquet",
        storage_options={"User-Agent": "byot-workshop"},
    )
    selected = points[points["class"] == CLASS]

px.scatter_geo(
    selected.head(MAP_LIMIT),
    lon="lon",
    lat="lat",
    hover_name="name",
    hover_data=[c for c in ("date",) if c in selected],
    title=CLASS.replace("_", " ").title(),
).show()

# 4. Choose your data

All collections below are available through [Microsoft Planetary Computer](https://planetarycomputer.microsoft.com), provide global coverage, and require no account. Choose one. Encoding preserves every pixel value.

| Collection | Data | Available | Resolution | Dtype | Bands |
|---|---|---|---|---|---|
| `sentinel-2-l2a` | Sentinel-2 surface reflectance | 2015-present | 10, 20, 60 m | uint16 | B01, B02, B03, B04, B05, B06, B07, B08, B8A, B09, B11, B12 |
| `landsat-c2-l2` | Landsat surface reflectance | 1982-present | 30 m | uint16 | coastal, blue, green, red, nir08, swir16, swir22 |
| `hls2-s30` | Harmonized Landsat Sentinel-2, Sentinel-2 component | 2020-present | 30 m | int16 | B01, B02, B03, B04, B05, B06, B07, B08, B8A, B09, B10, B11, B12 |
| `hls2-l30` | Harmonized Landsat Sentinel-2, Landsat component | 2020-present | 30 m | int16 | B01, B02, B03, B04, B05, B06, B07, B09, B10, B11 |
| `alos-palsar-mosaic` | ALOS PALSAR annual radar mosaic | 2015-2021 | 25 m | uint16 | HH, HV |
| `nasadem` | NASADEM elevation | 2000 | 30 m | int16 | elevation |
| `alos-dem` | ALOS World 3D elevation | 2016 | 30 m | int16 | data |
| `hgb` | Harmonized Global Biomass | 2010 | 300 m | uint16 | aboveground, belowground, aboveground_uncertainty, belowground_uncertainty |
| `alos-fnf-mosaic` | ALOS forest/non-forest map | 2015-2020 | 25 m | uint8 | C |
| `esa-worldcover` | ESA WorldCover land cover | 2020-2021 | 10 m; input_quality 60 m | map uint8; input_quality int16 | map, input_quality |
| `io-lulc-annual-v02` | Esri annual land use/land cover | 2017-2023 | 10 m | uint8 | data |
| `esa-cci-lc` | ESA CCI land cover | 1992-2020 | 300 m | uint8; observation_count uint16 | lccs_class, change_count, observation_count, current_pixel_state, processed_flag |
| `jrc-gsw` | JRC Global Surface Water | 1984-2020 | 30 m | uint8 | occurrence, change, seasonality, recurrence, transitions, extent |


### From points to Major TOM samples

Every sample is one **Major TOM cell**. Your points only say *which* cells: all points that fall in the same cell become one sample. The sample covers the whole cell plus a small margin, in the cell's UTM zone, on the product's own pixel grid. At the default spacing of 10 km (Major TOM Core), a Sentinel-2 sample is 10.56 km wide: 1056 pixels at 10 m.

- `GRID_KM` sets the grid spacing. Any value works; 10 km matches Major TOM Core and the GeoEnrich lookup in section 6.
- Products stored in latitude/longitude (the elevation, land-cover, water, biomass and ALOS layers) are resampled onto the cell's UTM grid; satellite images keep their original pixels.
- Instead of points, you can take every cell in an area with `build.cells_in`.
- To give each sample its own dates or size, add columns to the cells table: `datetime` (a range `"2024-06-01/2024-06-30"` or one date), `days` (search that many days around the date) and `margin` (extra width as a fraction of the cell).

### Which scene for each sample

For satellite images, several acquisitions usually cover a cell. Two settings decide which one is used, and every sample records them:

- `prefer` is the order in which scenes are tried: `"clearest"` (default), `"cloudiest"`, `"nearest"` to each sample's date (or the middle of its range), or `"newest"`.
- `cloud` is the share of cloudy pixels you accept in the window, as a range: `(0, 0.05)` (default) for clear samples, `(0.3, 1.0)` for cloudy ones, or `None` for any amount.

If none of the scenes tried is in the range, the sample uses the closest one and records its real cloud share; with `strict=True` the sample is skipped instead.

Dates mean different things per collection: they pick **acquisitions** for Sentinel-2, Landsat and HLS; they pick the **year** for yearly products (the ALOS mosaics, WorldCover, Esri and ESA CCI land cover), using the newest year if none falls in your dates; and they are ignored for single-version products (NASADEM, ALOS DEM, biomass, JRC water).


In [ ]:
#@title Major TOM builder
# The download helpers now live in the majortom package: https://github.com/ESA-PhiLab/Major-TOM
from majortom import build


In [ ]:
COLLECTION = "sentinel-2-l2a"
DATETIME = "2024-01-01/2024-12-31"
N_SAMPLES = 30
PICK = "random"  # "random": a random choice of points; "top": the first N_SAMPLES samples, by "score" if there is one, else in file order
GRID_KM = 10   # Major TOM grid spacing in km; 10 km is Major TOM Core
FORCE = False  # True rebuilds samples that are already on disk

if PICK == "top":
    ranked = selected.sort_values("score", ascending=False, kind="stable") if "score" in selected else selected
    cells = build.cells_from_points(ranked, d=GRID_KM).head(N_SAMPLES)   # N distinct samples, best first
else:
    sample_points = selected.sample(min(N_SAMPLES, len(selected)), random_state=42)
    cells = build.cells_from_points(sample_points, d=GRID_KM)   # one sample per Major TOM cell (and date)

# Or every cell in an area (give the area a name for the dataset):
# cells = build.cells_in(bbox=(-112.0, 40.4, -111.5, 40.8), d=GRID_KM).assign(**{"class": "wasatch"})

# Or your own dates per sample, e.g. +- 15 days around a date:
# cells = cells.assign(datetime="2024-07-01", days=15)

# Which scene: the order to try them in, and the share of cloudy pixels you accept (see above)
PREFER = "clearest"   # "clearest", "cloudiest", "nearest" or "newest"
CLOUD = (0, 0.05)     # e.g. (0.3, 1.0) for cloudy samples, None for any

errors = build.download(cells, COLLECTION, DATETIME, prefer=PREFER, cloud=CLOUD, force=FORCE)
errors


In [ ]:
#@title Look at your samples
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

# true-colour bands and brightness range (low, span) per optical collection; others show their first band
RGB = {"sentinel-2-l2a": (["B04", "B03", "B02"], 1000, 3000),   # +1000 offset from 25 Jan 2022 on
       "landsat-c2-l2": (["red", "green", "blue"], 7000, 12000),  # Collection 2 scale: 7273 is 0 reflectance
       "hls2-s30": (["B04", "B03", "B02"], 0, 2500),
       "hls2-l30": (["B04", "B03", "B02"], 0, 2500)}


def picture(collection, z, when):
    """One sample as an image: true colour with a fixed stretch, or the first band stretched 2-98%."""
    if collection in RGB:
        bands, low, span = RGB[collection]
        if collection == "sentinel-2-l2a" and when < "2022-01-25":
            low = 0                                          # older scenes have no offset
        rgb = np.stack([z[b].astype(np.float32) for b in bands], axis=-1)
        return np.clip((rgb - low) / span, 0, 1) ** 0.7, None   # gamma 0.7 brightens dark land
    band = z[z.files[0]].astype(np.float32)
    low, high = np.percentile(band, [2, 98])
    return np.clip((band - low) / max(high - low, 1e-6), 0, 1), "terrain"


sidecars = [json.loads(p.read_text()) for p in sorted(Path("chips", COLLECTION).glob("*.json"))][:12]
rows = max(1, -(-len(sidecars) // 4))   # 4 per row, only as many rows as needed
fig, axes = plt.subplots(rows, 4, figsize=(14, 3.8 * rows), squeeze=False, layout="constrained")
for ax in axes.flat:
    ax.axis("off")
for ax, meta in zip(axes.flat, sidecars):
    image, cmap = picture(COLLECTION, np.load(Path("chips", COLLECTION, f"{meta['index']:06d}.npz")), meta["datetime"])
    ax.imshow(image, cmap=cmap)
    cloud = "" if meta["cloud"] is None else f" · {meta['cloud']:.0%} cloud"
    ax.set_title(f"{meta['grid_cell']}\n{meta['datetime'][:10]}{cloud}", fontsize=9)
fig.suptitle(f"{COLLECTION}: {len(sidecars)} samples, cloud accepted {CLOUD}")

# 5. Encode with Rumi

Bands on the same spatial grid are stored together. For example, a Sentinel-2 sample produces `sentinel2-l2a-10m.rumi`, `sentinel2-l2a-20m.rumi`, and `sentinel2-l2a-60m.rumi`.

A `.rumi` file contains independently compressed frames. TACO stores its small index as a header in the corresponding row, so a reader can find the required byte range before opening the file.

Rumi requires a **name for every band** and a **time for every step**. Before encoding, set the band descriptions in `BAND_NAMES`. The download step provides the acquisition times in `dates`. GeoTIFF and TACO use the same values.

The workflow also writes each group as a multiband GeoTIFF for the size and speed comparison. Set its compression method with `COMPRESS`.


In [ ]:
#@title Rumi helpers { display-mode: "form" }
import json
from pathlib import Path

import geozl
import numpy as np
import pandas as pd
import rumi
from tqdm.auto import tqdm


RASTER_GROUPS = {
    "sentinel-2-l2a": {
        "sentinel2-l2a-10m": ["B02", "B03", "B04", "B08"],
        "sentinel2-l2a-20m": ["B05", "B06", "B07", "B8A", "B11", "B12"],
        "sentinel2-l2a-60m": ["B01", "B09"],
    },
    "landsat-c2-l2": {
        "landsat-c2-l2-30m": ["coastal", "blue", "green", "red", "nir08", "swir16", "swir22"],
    },
    "hls2-s30": {
        "hls-s30-30m": ["B01", "B02", "B03", "B04", "B05", "B06", "B07", "B08", "B8A", "B09", "B10", "B11", "B12"],
    },
    "hls2-l30": {
        "hls-l30-30m": ["B01", "B02", "B03", "B04", "B05", "B06", "B07", "B09", "B10", "B11"],
    },
    "alos-palsar-mosaic": {"alos-palsar-25m": ["HH", "HV"]},
    "nasadem": {"nasadem-30m": ["elevation"]},
    "alos-dem": {"alos-world3d-30m": ["data"]},
    "hgb": {
        "hgb-300m": ["aboveground", "belowground", "aboveground_uncertainty", "belowground_uncertainty"],
    },
    "alos-fnf-mosaic": {"alos-fnf-25m": ["C"]},
    "esa-worldcover": {
        "esa-worldcover-10m": ["map"],
        "esa-worldcover-quality-60m": ["input_quality"],
    },
    "io-lulc-annual-v02": {"io-lulc-10m": ["data"]},
    "esa-cci-lc": {
        "esa-cci-lc-300m": ["lccs_class", "change_count", "observation_count", "current_pixel_state", "processed_flag"],
    },
    "jrc-gsw": {
        "jrc-gsw-30m": ["occurrence", "change", "seasonality", "recurrence", "transitions", "extent"],
    },
}


def resolve_groups(collection, available):
    """Expand configured asset names to the planes present in one downloaded chip."""
    groups, assigned = {}, set()
    for group, assets in RASTER_GROUPS[collection].items():
        bands = []
        for asset in assets:
            matches = [name for name in available if name == asset or name.startswith(f"{asset}_")]
            if not matches:
                raise KeyError(f"{asset!r} is missing from {collection}")
            bands.extend(matches)
        groups[group] = bands
        assigned.update(bands)
    unassigned = set(available) - assigned
    if unassigned:
        raise KeyError(f"no raster group for {sorted(unassigned)}")
    return groups


def group_array(source, bands):
    arrays = [source[band] for band in bands]
    shapes = {array.shape for array in arrays}
    if len(shapes) != 1:
        raise ValueError(f"bands in one raster group have different shapes: {sorted(shapes)}")
    dtype = np.result_type(*(array.dtype for array in arrays))
    return np.stack([array.astype(dtype, copy=False) for array in arrays])


def to_rumi(collection, band_names, dates, graph="planar>zigzag>pfor"):
    """Write one multiband Rumi file for each configured spatial grid."""
    rows = []
    chips = sorted(Path("chips", collection).glob("*.npz"))
    if not chips:
        raise FileNotFoundError(f"no chips in chips/{collection}; run the download cell first")

    for npz in tqdm(chips, desc=graph):
        meta = json.loads(npz.with_suffix(".json").read_text())
        with np.load(npz) as z:
            for group, bands in resolve_groups(collection, meta["bands"]).items():
                array = group_array(z, bands)
                frames = rumi.frames(array, "b (row h) (col w) -> row col (b h w)", tile_size=array.shape[-1])
                for frame in frames:
                    frame.compressed = geozl.compress(frame.data, graph=geozl.graph(frame.data, graph))

                res = meta["res"] * meta["size"] / array.shape[-1]
                path = Path("rumi", collection, f"{meta['index']:06d}", f"{group}.rumi")
                path.parent.mkdir(parents=True, exist_ok=True)
                path, header = rumi.write(
                    path,
                    frames,
                    bands=[band_names[band] for band in bands],
                    time=dates[meta["index"]],
                    transform=(res, 0.0, meta["left"], 0.0, -res, meta["top"]),
                    crs=meta["epsg"],
                )
                rows.append(dict(index=meta["index"], band=group, bands=tuple(bands), path=str(path), header=header,
                                 graph=graph, res=res, width=array.shape[-1],
                                 dtype=str(array.dtype), rumi_bytes=Path(path).stat().st_size))

    index = pd.DataFrame(rows)
    return index

In [ ]:
#@title Plot helpers { display-mode: "form" }
import matplotlib.pyplot as plt


def show_roundtrip(index, collection, sample=None, bands=None, limit=6):
    """Show original and decoded values for a few bands of one chip."""
    sample = int(index["index"].iloc[0]) if sample is None else int(sample)
    rows = index[index["index"] == sample]
    if not len(rows):
        raise KeyError(f"sample {sample} is not in this index")
    selected = None if bands is None else set(bands)
    panels = []
    with np.load(Path("chips", collection, f"{sample:06d}.npz")) as z:
        for row in rows.itertuples():
            decoded = np.asarray(rumi.read(row.path, row.header))
            for position, band in enumerate(row.bands):
                if selected is None or band in selected:
                    panels.append((band, z[band], decoded[position]))
                if len(panels) == limit:
                    break
            if len(panels) == limit:
                break

    fig, axes = plt.subplots(len(panels), 2, figsize=(7, 3 * len(panels)), squeeze=False)
    for ax, (band, before, after) in zip(axes, panels):
        lo, hi = np.percentile(before, [2, 98])
        if lo == hi:  # a constant band would make imshow complain
            lo, hi = lo - 0.5, hi + 0.5
        ax[0].imshow(before, cmap="gray", vmin=lo, vmax=hi)
        ax[1].imshow(after, cmap="gray", vmin=lo, vmax=hi)
        ax[0].set_ylabel(band)
        for a in ax:
            a.set_xticks([])
            a.set_yticks([])
    axes[0, 0].set_title("original")
    axes[0, 1].set_title("stored")
    fig.tight_layout()
    plt.show()

In [ ]:
#@title GeoTIFF helpers { display-mode: "form" }
import rasterio
from rasterio.crs import CRS
from rasterio.transform import from_origin


def to_geotiff(collection, band_names, compress="zstd"):
    """Write one multiband GeoTIFF for each configured spatial grid.

    compress is any GDAL creation option, so zstd, deflate, lzw or None.
    """
    rows = []
    chips = sorted(Path("chips", collection).glob("*.npz"))
    if not chips:
        raise FileNotFoundError(f"no chips in chips/{collection}; run the download cell first")
    options = dict(driver="GTiff")
    if compress:
        options.update(compress=compress, predictor=2)

    for npz in tqdm(chips, desc=f"geotiff {compress}"):
        meta = json.loads(npz.with_suffix(".json").read_text())
        with np.load(npz) as z:
            for group, bands in resolve_groups(collection, meta["bands"]).items():
                array = group_array(z, bands)
                res = meta["res"] * meta["size"] / array.shape[-1]
                path = Path("geotiff", collection, f"{meta['index']:06d}", f"{group}.tif")
                path.parent.mkdir(parents=True, exist_ok=True)
                with rasterio.open(path, "w", count=array.shape[0], height=array.shape[1], width=array.shape[2],
                                   dtype=array.dtype, crs=CRS.from_epsg(meta["epsg"]),
                                   transform=from_origin(meta["left"], meta["top"], res, res),
                                   **options) as dst:
                    dst.write(array)
                    for position, band in enumerate(bands, start=1):
                        dst.set_band_description(position, band_names[band])
                rows.append(dict(index=meta["index"], band=group, bands=tuple(bands), path=str(path), header=b"",
                                 graph=f"GTiff {compress or 'uncompressed'}",
                                 res=res, width=array.shape[-1], dtype=str(array.dtype),
                                 rumi_bytes=path.stat().st_size))

    index = pd.DataFrame(rows)
    return index

In [ ]:
#@title Band metadata helpers { display-mode: "form" }
from datetime import datetime

BAND_NAMES = {
    "sentinel-2-l2a": {
        "B01": "B01, Coastal aerosol, 443nm", "B02": "B02, Blue, 490nm", "B03": "B03, Green, 560nm",
        "B04": "B04, Red, 665nm", "B05": "B05, Red edge 1, 705nm", "B06": "B06, Red edge 2, 740nm",
        "B07": "B07, Red edge 3, 783nm", "B08": "B08, NIR, 842nm", "B8A": "B8A, Narrow NIR, 865nm",
        "B09": "B09, Water vapour, 945nm", "B11": "B11, SWIR 1, 1610nm", "B12": "B12, SWIR 2, 2190nm",
    },
    "landsat-c2-l2": {
        "coastal": "coastal, Coastal aerosol, 443nm", "blue": "blue, Blue, 482nm", "green": "green, Green, 561nm",
        "red": "red, Red, 655nm", "nir08": "nir08, NIR, 865nm", "swir16": "swir16, SWIR 1, 1609nm",
        "swir22": "swir22, SWIR 2, 2201nm",
    },
    "hls2-s30": {
        "B01": "B01, Coastal aerosol, 443nm", "B02": "B02, Blue, 490nm", "B03": "B03, Green, 560nm",
        "B04": "B04, Red, 665nm", "B05": "B05, Red edge 1, 705nm", "B06": "B06, Red edge 2, 740nm",
        "B07": "B07, Red edge 3, 783nm", "B08": "B08, NIR broad, 842nm", "B8A": "B8A, NIR narrow, 865nm",
        "B09": "B09, Water vapour, 945nm", "B10": "B10, Cirrus, 1375nm", "B11": "B11, SWIR 1, 1610nm",
        "B12": "B12, SWIR 2, 2190nm",
    },
    "hls2-l30": {
        "B01": "B01, Coastal aerosol, 443nm", "B02": "B02, Blue, 482nm", "B03": "B03, Green, 561nm",
        "B04": "B04, Red, 655nm", "B05": "B05, NIR narrow, 865nm", "B06": "B06, SWIR 1, 1609nm",
        "B07": "B07, SWIR 2, 2201nm", "B09": "B09, Cirrus, 1373nm", "B10": "B10, Thermal 1, 10.9um",
        "B11": "B11, Thermal 2, 12.0um",
    },
    "alos-palsar-mosaic": {"HH": "HH, L-band backscatter, HH polarization", "HV": "HV, L-band backscatter, HV polarization"},
    "nasadem": {"elevation": "elevation, Height in metres"},
    "alos-dem": {"data": "elevation, ALOS World 3D height in metres"},
    "hgb": {
        "aboveground": "aboveground, Aboveground biomass carbon density",
        "belowground": "belowground, Belowground biomass carbon density",
        "aboveground_uncertainty": "aboveground_uncertainty, Aboveground biomass uncertainty",
        "belowground_uncertainty": "belowground_uncertainty, Belowground biomass uncertainty",
    },
    "alos-fnf-mosaic": {"C": "C, Forest / non-forest class"},
    "esa-worldcover": {
        "map": "map, Land cover class",
        "input_quality_1": "input_quality_1, Sentinel-1 observations",
        "input_quality_2": "input_quality_2, Sentinel-2 observations",
        "input_quality_3": "input_quality_3, Invalid Sentinel-2 observations, percent",
    },
    "io-lulc-annual-v02": {"data": "land_cover, Land use / land cover class"},
    "esa-cci-lc": {
        "lccs_class": "lccs_class, Land cover class", "change_count": "change_count, Number of class changes",
        "observation_count": "observation_count, Number of valid observations",
        "current_pixel_state": "current_pixel_state, Pixel state", "processed_flag": "processed_flag, Processing flag",
    },
    "jrc-gsw": {
        "occurrence": "occurrence, Water occurrence, percent", "change": "change, Occurrence change intensity",
        "seasonality": "seasonality, Months with water", "recurrence": "recurrence, Water recurrence, percent",
        "transitions": "transitions, Water class transitions", "extent": "extent, Maximum water extent",
    },
}

chips = [json.loads(path.read_text()) for path in sorted(Path("chips", COLLECTION).glob("*.json"))]
missing = sorted(set(chips[0]["bands"]) - set(BAND_NAMES.get(COLLECTION, {})))
if missing:
    raise KeyError(f"BAND_NAMES[{COLLECTION!r}] has no name for {missing}")

# rumi stores whole seconds
dates = {chip["index"]: [datetime.fromisoformat(chip["datetime"]).replace(microsecond=0)] for chip in chips}

{band: BAND_NAMES[COLLECTION][band] for band in chips[0]["bands"]}, dates[chips[0]["index"]]

In [ ]:
GRAPH = "planar>zigzag>pfor"  # GeoZL graph used for every raster group
COMPRESS = "zstd"  # GeoTIFF compression

geotiff = to_geotiff(COLLECTION, BAND_NAMES[COLLECTION], COMPRESS)
pfor = to_rumi(COLLECTION, BAND_NAMES[COLLECTION], dates, GRAPH)

# Average size of each raster group.
groups = pfor.groupby("band")
sizes = pd.DataFrame({
    "bands": groups["bands"].first().map(len),
    "pixels": groups["width"].first().map(lambda width: f"{width}x{width}"),
    "dtype": groups["dtype"].first(),
    "geotiff_kb": geotiff.groupby("band")["rumi_bytes"].mean() / 1024,
    "rumi_kb": groups["rumi_bytes"].mean() / 1024,
})
sizes["ratio"] = sizes["geotiff_kb"] / sizes["rumi_kb"]
sizes.rename_axis(None).round({"geotiff_kb": 1, "rumi_kb": 1, "ratio": 2})

In [ ]:
show_roundtrip(pfor, COLLECTION)

# 6. Package as TACO

A TACO dataset starts with a **contract**. It defines the files and metadata in every sample and is fixed before writing begins. The writer rejects samples that do not match it. Readers can use the metadata to filter samples before opening a raster.

### Files

Each sample stores the same raster groups in two formats:

```
rumi/<group>.rumi       Rumi with lossless GeoZL compression
geotiff/<group>.tif     compressed GeoTIFF with the same pixels
```

### Levels

Metadata is stored in **levels**, with one table for each type of row. TACO derives the levels from the file structure. A `children/<folder>` level has one row for each file in that folder.

| Level | One row per | Stores |
|---|---|---|
| `sample` | chip | grid, acquisition time, Major TOM cell, and environmental context |
| `children/rumi` | `.rumi` file | Rumi header and per-band statistics |
| `children/geotiff` | `.tif` file | no extra metadata; the path is sufficient |

Dataset-wide facts are stored once in `COLLECTION.json` as **collection metadata**.

### Metadata groups

Each keyword in a level defines a **group**. Fields use names such as `stac:centroid` and `majortom:code`. Most groups below are **extensions** that TACO computes while writing.

| Group | Where | You supply | TACO computes |
|---|---|---|---|
| `stac` | `sample` | grid (`proj_code`, `proj_shape`, `proj_transform`) and `datetime` | `centroid`, the grid center in longitude and latitude |
| `majortom` | `sample` | nothing | `code`, the 10 km Major TOM cell containing the centroid |
| `geoenrich` | `sample` | nothing | elevation, climate, population, human modification, and administrative names |
| `rumi` | `children/rumi` | nothing | `header` and per-band `stats`, read from each file |
| `poi` | collection | Wikidata class | nothing |
| `source` | collection | Planetary Computer collection | nothing |

`stac` requires **where** and **when**: a grid (or `geometry`) and either `datetime` or a `start_datetime`/`end_datetime` range. `sample_metadata` uses the same grid and times as the Rumi files. One time becomes `datetime`; several become a range.

The extensions run in this order:

```
stac:centroid  ->  majortom:code  ->  geoenrich:*
```

GeoEnrich looks up each code in the public 10 km Major TOM index on Source Cooperative, so `MajorTOM(dist_km=10)` must use the same grid. No Earth Engine account is required.

### Your turn

Edit the next cell. `build_collection` defines the contract, and `sample_metadata` provides values for one chip. To add a field, declare its group in a level and provide its value. For example, to add a train/test split:

```python
from pydantic import BaseModel, Field

class ML(BaseModel):
    split: str = Field(description="Dataset split")

taco.Level("sample", ..., ml=ML)                               # in build_collection
ml=ML(split="test" if meta["index"] % 5 == 0 else "train")     # in sample_metadata
```

If the declaration and value do not match, the writer stops at the first invalid sample and names the group.

Add dataset-wide facts to `taco.Collection` in `build_collection`, as with `poi` and `source`. For example: `region={"name": "Andes"}`.


In [ ]:
# Your contract. This is the cell to edit.

import taco


def build_collection(info):
    """The contract and the description of the dataset."""
    contract = taco.Contract(
        structure=([f"rumi/{group}.rumi" for group in info["groups"]]
                   + [f"geotiff/{group}.tif" for group in info["groups"]]),
        metadata=[
            taco.Level(
                "sample",
                stac=taco.extensions.STAC(),
                majortom=taco.extensions.MajorTOM(dist_km=GRID_KM),  # GeoEnrich below needs 10 km
                geoenrich=taco.extensions.GeoEnrich([
                    "elevation",
                    "temperature",
                    "precipitation",
                    "population",
                    "human_modification",
                    "admin_countries",
                    "admin_states",
                    "admin_districts",
                ]),
            ),
            taco.Level("children/rumi", rumi=taco.extensions.Rumi(stats=True)),
        ],
    )
    return taco.Collection(
        contract=contract,
        id=info["id"],
        description=(
            f"{info['collection']} samples on the {GRID_KM} km Major TOM grid, one per cell containing "
            f"{info['category']} points from Wikidata. "
            f"Every raster group twice, as lossless {info['graph']} Rumi and as a "
            f"Zstandard-compressed GeoTIFF."
        ),
        licenses=["CC-BY-4.0"],
        providers=[
            {"name": info["author"], "roles": ["producer"]},
            {"name": "Microsoft Planetary Computer", "roles": ["host"]},
        ],
        tasks=["classification"],
        keywords=[info["collection"], info["category"], "major-tom"],
        poi={"category": info["category"], "catalog": "Wikidata"},
        source={"collection": info["collection"]},
    )


def sample_metadata(meta, times):
    """The values you supply for one chip: its JSON from the download and its times."""
    first, last = min(times), max(times)
    when = {"datetime": first} if first == last else {"start_datetime": first, "end_datetime": last}
    return taco.Metadata(
        stac=taco.metadata.sample.STAC(
            proj_code=f"EPSG:{meta['epsg']}",
            proj_shape=(meta["size"], meta["size"]),
            proj_transform=(meta["res"], 0.0, meta["left"], 0.0, -meta["res"], meta["top"]),
            **when,
        ),
    )


sample_metadata(chips[0], dates[chips[0]["index"]]).groups["stac"]

In [ ]:
#@title TACO helpers { display-mode: "form" }
import json
import re
from pathlib import Path

import taco


def slug(text):
    return re.sub(r"[^a-z0-9]+", "-", str(text).lower()).strip("-")


def to_taco(
    indexes,
    collection,
    points,
    author,
    graph,
    dates,
    build=build_collection,
    values=sample_metadata,
):
    """One TACO ZIP with every raster group stored in both formats."""
    folders = {"rumi": ".rumi", "geotiff": ".tif"}

    # Read the sample IDs and raster groups produced by the previous steps.
    samples = sorted(indexes["rumi"]["index"].unique())
    groups = tuple(indexes["rumi"].loc[indexes["rumi"]["index"] == samples[0], "band"])

    # Make every generated file easy to find by sample and raster group.
    indexes = {folder: index.set_index(["index", "band"]) for folder, index in indexes.items()}

    # Use the selected class and collection to create a readable dataset identity.
    category = points.loc[samples[0], "class"]
    name = slug(f"byot {author} {category} {collection}")

    # Build one contract before writing any samples.
    dataset = build({
        "groups": groups,
        "id": name,
        "collection": collection,
        "category": category,
        "graph": graph,
        "author": author,
    })

    # Package each chip, its metadata and both file formats into one TACO ZIP.
    output = Path("taco", f"{name}.zip")
    output.parent.mkdir(parents=True, exist_ok=True)
    with taco.open_writer(dataset, output, overwrite=True, progress=True) as writer:
        for sample in samples:
            meta = json.loads(Path("chips", collection, f"{sample:06d}.json").read_text())

            # Every asset receives the path declared by the contract.
            assets = [
                taco.Asset(
                    Path(indexes[folder].loc[(sample, group)].path),
                    path=f"{folder}/{group}{suffix}",
                )
                for folder, suffix in folders.items()
                for group in groups
            ]
            writer.add(taco.Sample(
                id=f"{sample:06d}",
                metadata=values(meta, dates[sample]),
                assets=assets,
            ))

        # run() commits the finished archive.
        writer.run()

    # Validate the published dataset before returning it.
    report = taco.validate(output)
    print(f"{output} | {len(samples)} samples x {len(groups)} raster groups x 2 formats | "
          f"{output.stat().st_size / 1e6:.1f} MB | valid={report.ok}")
    return output

In [ ]:
AUTHOR = "Your Name"

dataset_zip = to_taco({"rumi": pfor, "geotiff": geotiff}, COLLECTION, cells,
                      AUTHOR, GRAPH, dates)

# 7. Publish on Hugging Face

A TACO dataset can be a folder or a single file. We created a ZIP, so we can upload it as one object. Readers can open its URL and fetch only the byte ranges they need.

1. Sign in at [huggingface.co](https://huggingface.co), or create an account.
2. Open [Settings → Access Tokens](https://huggingface.co/settings/tokens) and create a token with **Write** access.
3. Paste the token into `HF_TOKEN` below and set `REPO` to the destination dataset repository.

The next cells create a public dataset repository under your account. Share the URL printed at the end.


In [ ]:
#@title Upload helpers { display-mode: "form" }
from pathlib import Path

from huggingface_hub import HfApi


def upload(path, repo, token):
    if not token:
        raise ValueError("paste a Hugging Face token with Write access into HF_TOKEN first")
    if repo.startswith("your-user/"):
        raise ValueError("set REPO to your-hugging-face-user/dataset-name")
    path = Path(path)
    api = HfApi(token=token)
    api.create_repo(repo, repo_type="dataset", private=False, exist_ok=True)
    api.upload_file(path_or_fileobj=str(path), path_in_repo=path.name, repo_id=repo, repo_type="dataset")
    url = f"https://huggingface.co/datasets/{repo}/resolve/main/{path.name}"
    print(url)
    return url

In [ ]:
HF_TOKEN = ""
REPO = "your-user/byot-dataset"

In [ ]:
dataset_url = upload(dataset_zip, REPO, HF_TOKEN)
taco.open_dataset(dataset_url)

# 8. Query your dataset

Open the published dataset by URL and query it.

- `dataset.read()` returns every sample as one table.
- `dataset.sql()` runs SQL on these relations:
  - `dataset`: one row per sample, with each file's `::location`, `::header` and statistics.
  - `sample`: the sample metadata.
  - `"children/rumi"`: one row per `.rumi` file, with its header and statistics.
- `rumi.read(location, header)` reads a chip.

In [ ]:
#@title Display helpers { display-mode: "form" }
import matplotlib.pyplot as plt
import numpy as np
import rumi

FILE = f"rumi/{pfor['band'].iloc[0]}.rumi"  # first raster group, e.g. sentinel2-l2a-10m
names = list(pfor["bands"].iloc[0])
RGB = [names.index(band) for band in ("B04", "B03", "B02")] if {"B04", "B03", "B02"} <= set(names) else [0, 0, 0]


def show_rgb(rows, title="id"):
    """Read each chip from its location and header, and plot it in true color."""
    rows = rows.to_dict("records")
    fig, axes = plt.subplots(1, len(rows), figsize=(3 * len(rows), 3), squeeze=False)
    for ax, row in zip(axes[0], rows):
        image = np.asarray(rumi.read(row["location"], row["header"]))[RGB].transpose(1, 2, 0).astype("float32")
        low, high = np.percentile(image, [2, 98])
        ax.imshow(np.clip((image - low) / max(high - low, 1), 0, 1))
        ax.set_title(str(row[title]))
        ax.axis("off")
    plt.show()

## 8.1 Which samples are in France?

`dataset.read()` loads all samples, then pandas selects those in France.

In [ ]:
import taco

dataset = taco.open_dataset(dataset_url)

samples = dataset.read().to_pandas()
france = samples[samples["geoenrich:admin_countries"] == "France"].rename(
    columns={f"{FILE}::location": "location", f"{FILE}::header": "header"}
)
show_rgb(france.head(4))
france[["id", "geoenrich:admin_states", "location", "header"]]

## 8.2 Which samples are at the highest and lowest elevation?

The query finds the elevation range in `sample` and returns the matching chips from `dataset`.

In [ ]:
extremes = dataset.sql(f"""
    SELECT id, round("geoenrich:elevation", 1) AS elevation,
           "{FILE}::location" AS location, "{FILE}::header" AS header
    FROM dataset
    WHERE "geoenrich:elevation" IN ((SELECT max("geoenrich:elevation") FROM sample),
                                    (SELECT min("geoenrich:elevation") FROM sample))
    ORDER BY elevation DESC
""").to_pandas()
show_rgb(extremes, title="elevation")
extremes

## 8.3 Which US chips have more texture than the rest of the dataset?

TACO stored statistics such as `stddev` for every `.rumi` file. This query selects US chips above the dataset's median texture without reading any pixels. The median comes from `"children/rumi"`, which has one row per file.

In [ ]:
textured = dataset.sql(f"""
    SELECT id, "geoenrich:admin_states" AS state, "{FILE}::stddev" AS std,
           "{FILE}::location" AS location, "{FILE}::header" AS header
    FROM dataset
    WHERE "geoenrich:admin_countries" = 'United States'
      AND "{FILE}::stddev" > (SELECT median("rumi:stddev") FROM "children/rumi"
                              WHERE "internal:relative_path" LIKE '%/{FILE}')
    ORDER BY std DESC
""").to_pandas()
show_rgb(textured, title="state")
textured

# 9. Explore your dataset in the TACO Playground

Open your published dataset in the [TACO Playground](https://asterisk.coop/taco/playground/) to map samples, inspect metadata, color points by a field, and filter with SQL.

Run the next cell, then click a point, choose a field under **Color points by**, or query the `dataset` or `sample` table under **SQL filter**.


In [ ]:
from urllib.parse import quote

from IPython.display import Markdown, display


playground_url = f"https://asterisk.coop/taco/playground/?url={quote(dataset_url, safe='')}"
display(Markdown(f"[Open your dataset in the TACO Playground]({playground_url})"))

# 10. Train directly from the dataset

Read all 1,000 chips once—one training epoch—and compare GeoTIFF with Rumi.

## 10.1 Download the dataset

aria2 downloads the 1 GB ZIP over 16 connections. Training then reads from local disk.

In [ ]:
# aria2 downloads the 1 GB dataset over 16 connections; training then reads from the local disk.
URL = "https://data.source.coop/asterisk-labs/cng2026/datasets/byot-golf-course-sentinel-2-l2a.zip"
!apt-get -qq install -y aria2 > /dev/null
!aria2c --quiet --continue -x 16 -s 16 -d data -o golf-course.zip {URL}
!ls -lh data/golf-course.zip

## 10.2 Build a table of chips

`dataset.sql(...).to_pandas()` returns one row per sample. Each `::location` is a GDAL `/vsisubfile/` path to the file's bytes inside the ZIP.

In [ ]:
import taco

dataset = taco.open_dataset("data/golf-course.zip")

rumi_chips = dataset.sql("""
    SELECT id,
           "rumi/sentinel2-l2a-10m.rumi::location" AS "10m",
           "rumi/sentinel2-l2a-10m.rumi::header" AS "10m_header",
           "rumi/sentinel2-l2a-20m.rumi::location" AS "20m",
           "rumi/sentinel2-l2a-20m.rumi::header" AS "20m_header",
           "rumi/sentinel2-l2a-60m.rumi::location" AS "60m",
           "rumi/sentinel2-l2a-60m.rumi::header" AS "60m_header"
    FROM dataset
""").to_pandas()

geotiff_chips = dataset.sql("""
    SELECT id,
           "geotiff/sentinel2-l2a-10m.tif::location" AS "10m",
           "geotiff/sentinel2-l2a-20m.tif::location" AS "20m",
           "geotiff/sentinel2-l2a-60m.tif::location" AS "60m"
    FROM dataset
""").to_pandas()

rumi_chips.head(3)

## 10.3 Create the PyTorch datasets

- `GeoTIFFDataset.__getitem__` reads one chip with rasterio. The `DataLoader` builds the batch.
- `RumiDataset.__getitems__` reads a full batch with one `rumi.read_many` call.

`read_many` takes the locations and headers for a batch, decodes them together in native code, and returns one tensor with the batch on the first axis. Since the batch is already built, `collate_fn` returns it unchanged.

In [ ]:
import statistics
import time

import rasterio
import rumi
import torch
from torch.utils.data import DataLoader, Dataset


RESOLUTIONS = ("10m", "20m", "60m")


class GeoTIFFDataset(Dataset):
    """Reads one chip; the DataLoader stacks the chips into a batch."""

    def __init__(self, chips):
        self.chips = chips.to_dict("records")  # one dict per row, fast to index

    def __len__(self):
        return len(self.chips)

    def __getitem__(self, index):
        chip = self.chips[index]
        tensors = {}
        for resolution in RESOLUTIONS:
            with rasterio.open(chip[resolution]) as src:
                tensors[resolution] = torch.from_numpy(src.read())
        return tensors


class RumiDataset(Dataset):
    """Reads a whole batch with one rumi.read_many call per resolution."""

    def __init__(self, chips):
        self.chips = chips.to_dict("records")  # one dict per row, fast to index

    def __len__(self):
        return len(self.chips)

    def __getitems__(self, indices):
        batch = [self.chips[index] for index in indices]
        tensors = {}
        for resolution in RESOLUTIONS:
            tensors[resolution] = rumi.read_many(
                [chip[resolution] for chip in batch],
                [chip[f"{resolution}_header"] for chip in batch],
                framework="torch",
            )
        return tensors


def make_loader(dataset, batch_size=64):
    # __getitems__ already returns a batch, so there is nothing left to collate.
    collate_fn = (lambda batch: batch) if hasattr(dataset, "__getitems__") else None
    return DataLoader(dataset, batch_size=batch_size, shuffle=True, num_workers=0, collate_fn=collate_fn)


def seconds_per_epoch(dataset, repeats=3):
    """Median time to read every chip once, so one slow epoch does not decide the result."""
    times = []
    for _ in range(repeats):
        start = time.perf_counter()
        for _batch in make_loader(dataset):
            pass
        times.append(time.perf_counter() - start)
    return statistics.median(times)

## 10.4 Time one epoch

The `DataLoader` reads all 1,000 chips in batches of 64. Each time is the median of three epochs.

In [ ]:
import pandas as pd

rumi_dataset = RumiDataset(rumi_chips)
geotiff_dataset = GeoTIFFDataset(geotiff_chips)

seconds = pd.Series({
    "GeoTIFF": seconds_per_epoch(geotiff_dataset),
    "Rumi": seconds_per_epoch(rumi_dataset),
})
pd.DataFrame({
    "reads": ["one chip (__getitem__)", "one batch (__getitems__)"],
    "seconds per epoch": seconds.round(2),
    "samples per second": (len(rumi_dataset) / seconds).round().astype(int),
    "speedup": (seconds["GeoTIFF"] / seconds).round(1),
})

# 11. Export a subset of a published dataset

`taco.export` copies samples selected by SQL into a new TACO dataset. It queries the metadata first and downloads only the selected files. The new dataset keeps the original contract and sample IDs.

## 11.1 Create the mini TACO

The query runs on [Major TOM Core-DEM](https://source.coop/major-tom/core-dem), a global elevation dataset with 1.7 million cells. It filters for cells in Peru above 4,000 m with little human modification, then selects the least modified cell in each region.

Keep `SELECT *`: `taco.export` needs `taco:sample_index` to find each sample.

In [ ]:
import taco


CORE_DEM = "https://data.source.coop/major-tom/core-dem"

QUERY = """
    SELECT *
    FROM sample
    WHERE "geoenrich:admin_countries" = 'Peru'
      AND "geoenrich:elevation" > 4000
      AND "geoenrich:human_modification" < 0.05
    QUALIFY row_number() OVER (
        PARTITION BY "geoenrich:admin_states"
        ORDER BY "geoenrich:human_modification", "geoenrich:elevation" DESC
    ) = 1
"""

result = taco.export(
    CORE_DEM,
    "taco/core-dem-peru-wild.zip",
    sql=QUERY,
    overwrite=True,
    id="core-dem-peru-wild",
    description="The least modified Major TOM Core-DEM cell above 4,000 m in each Peruvian region.",
)

## 11.2 Validate the mini TACO

`taco.validate` checks the new ZIP:

- metadata columns, types, and nulls match the contract;
- each `id` is unique, and each geometry and projection is valid;
- the ZIP index comes first, and each listed file exists at its recorded offset.

It returns a report with every error and warning. `report.ok` is `True` when there are no errors.

In [ ]:
report = taco.validate(result.path)
print(f"{result.samples} samples | {result.size / 1e6:.1f} MB")
print(report)

## 11.3 Open it and look at each region

The mini TACO opens like any other dataset. SQL on `dataset` returns the region, elevation, and `dem.rumi` location and header of each sample, and `rumi.read` loads its elevation model.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import rumi
from matplotlib.colors import LightSource, ListedColormap, Normalize

mini = taco.open_dataset(result.path)
regions = mini.sql("""
    SELECT "geoenrich:admin_states" AS region,
           "geoenrich:elevation" AS elevation,
           "dem.rumi::location" AS location,
           "dem.rumi::header" AS header
    FROM dataset
    ORDER BY elevation DESC
""").to_pandas()

light = LightSource(azdeg=315, altdeg=45)
land = ListedColormap(plt.cm.terrain(np.linspace(0.25, 1, 256)))  # terrain without its sea blues
norm = Normalize(vmin=2000, vmax=5500)
fig, axes = plt.subplots(3, 5, figsize=(15, 10), layout="constrained")
for ax, row in zip(axes.flat, regions.itertuples()):
    dem = np.asarray(rumi.read(row.location, row.header))[0].astype("float32")
    ax.imshow(light.shade(dem, cmap=land, norm=norm, vert_exag=0.05, blend_mode="soft"))
    ax.set_title(f"{row.region}\n{row.elevation:,.0f} m", fontsize=11)
    ax.axis("off")
fig.colorbar(plt.cm.ScalarMappable(norm, land), ax=axes, shrink=0.6, label="elevation (m)")
fig.suptitle("The least modified cell above 4,000 m in each region of Peru", fontsize=15)
plt.show()